# 06 — Grad-CAM 시각화
> **목표:** stage2 멀티태스크 모델로 Real/Fake 판단 근거를 얼굴 위 히트맵으로 시각화  
> **입력:** `stage2_best.h5` + 샘플 이미지 몇 장  
> **출력:** 공격 유형별 Grad-CAM 히트맵 비교 이미지

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

BASE       = '/content/drive/MyDrive/face-anti-spoofing'
MODEL_DIR  = f'{BASE}/models'
CROP_DIR   = f'{BASE}/data/cropped'
REPORT_DIR = f'{BASE}/reports'
os.makedirs(REPORT_DIR, exist_ok=True)

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. 모델 로드

In [ ]:
model = tf.keras.models.load_model(f'{MODEL_DIR}/stage2_best.h5')
model.summary(line_length=80)

# Grad-CAM 대상: MobileNetV2 마지막 conv 레이어
# 레이어 이름 확인
conv_layers = [l.name for l in model.layers if 'conv' in l.name.lower()]
print('\n마지막 Conv 레이어:', conv_layers[-1])
TARGET_LAYER = conv_layers[-1]

## 2. 전처리 함수

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess(img_bgr):
    """BGR 이미지 → 모델 입력 텐서 (1, 224, 224, 3)"""
    img = cv2.resize(img_bgr, (224, 224))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = (img.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
    return np.expand_dims(img, 0)

def load_img(fp):
    """Drive 경로 안전 로드 (imdecode 방식)"""
    raw = np.fromfile(str(fp), dtype=np.uint8)
    return cv2.imdecode(raw, cv2.IMREAD_COLOR)

print('전처리 함수 정의 완료')

## 3. Grad-CAM 함수

In [ ]:
def get_gradcam(model, img_array, target_layer_name, head_name='head_binary'):
    """
    img_array: (1, 224, 224, 3) 정규화된 텐서
    반환: heatmap (224, 224) 0~1
    """
    # 타겟 레이어 출력 + 모델 출력을 동시에 뽑는 서브모델
    grad_model = tf.keras.Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(target_layer_name).output,
                   model.get_layer(head_name).output]
    )

    with tf.GradientTape() as tape:
        inputs = tf.cast(img_array, tf.float32)
        conv_output, predictions = grad_model(inputs)
        # binary head: sigmoid 출력 → Spoof 확률
        loss = predictions[:, 0]

    # conv 출력에 대한 gradient
    grads = tape.gradient(loss, conv_output)

    # Global Average Pooling → 채널별 중요도
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # 가중합
    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # ReLU + 정규화
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_heatmap(img_bgr, heatmap, alpha=0.4):
    """원본 이미지에 히트맵 오버레이 → RGB 반환"""
    heatmap_resized = cv2.resize(heatmap, (img_bgr.shape[1], img_bgr.shape[0]))
    heatmap_colored = cv2.applyColorMap(
        np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
    )
    overlaid = cv2.addWeighted(
        cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), 1 - alpha,
        cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB), alpha, 0
    )
    return overlaid

print('Grad-CAM 함수 정의 완료')

## 4. 샘플 로드 (카테고리당 3장)

In [ ]:
CATEGORIES = {
    'live'  : {'label': 'Live (Real)', 'spoof_id': 0},
    'print' : {'label': 'Print Attack', 'spoof_id': 1},
    'replay': {'label': 'Replay Attack', 'spoof_id': 2},
    'mask'  : {'label': '3D Mask Attack', 'spoof_id': 3},
}
N_SAMPLES = 3  # 카테고리당 샘플 수

samples = {}  # {cat: [img_bgr, ...]}

for cat in CATEGORIES:
    cat_dir = Path(CROP_DIR) / cat
    files   = sorted(cat_dir.glob('*.jpg'))[:N_SAMPLES]
    imgs    = []
    for fp in files:
        img = load_img(fp)
        if img is not None:
            imgs.append(img)
    samples[cat] = imgs
    print(f'  {cat}: {len(imgs)}장 로드')

print('샘플 로드 완료')

## 5. Grad-CAM 시각화 — 공격 유형별 비교

In [ ]:
fig, axes = plt.subplots(
    len(CATEGORIES), N_SAMPLES * 2,
    figsize=(N_SAMPLES * 6, len(CATEGORIES) * 3)
)

for row, (cat, info) in enumerate(CATEGORIES.items()):
    for col, img_bgr in enumerate(samples[cat]):
        # 전처리 & 예측
        inp         = preprocess(img_bgr)
        pred_binary = model.predict(inp, verbose=0)
        # 멀티태스크: 첫 번째 출력이 binary
        if isinstance(pred_binary, list):
            spoof_prob = float(pred_binary[0][0][0])
        else:
            spoof_prob = float(pred_binary[0][0])
        verdict = 'FAKE' if spoof_prob >= 0.5 else 'REAL'

        # Grad-CAM
        heatmap  = get_gradcam(model, inp, TARGET_LAYER)
        overlaid = overlay_heatmap(img_bgr, heatmap)

        # 원본
        ax_orig = axes[row][col * 2]
        ax_orig.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        ax_orig.axis('off')
        if col == 0:
            ax_orig.set_title(f'{info["label"]}\n원본', fontsize=10, fontweight='bold')
        else:
            ax_orig.set_title('원본', fontsize=9)

        # Grad-CAM 오버레이
        ax_cam = axes[row][col * 2 + 1]
        ax_cam.imshow(overlaid)
        ax_cam.axis('off')
        color = 'red' if verdict == 'FAKE' else 'green'
        ax_cam.set_title(
            f'Grad-CAM\n{verdict} ({spoof_prob:.0%})',
            fontsize=9, color=color
        )

plt.suptitle('공격 유형별 Grad-CAM — 판단 근거 시각화', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/06_gradcam_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장 완료: 06_gradcam_comparison.png')

## 6. 단일 이미지 상세 분석 (발표용)

In [ ]:
def analyze_single(img_bgr, title=''):
    """원본 / Grad-CAM / 히트맵만 3단 비교"""
    inp = preprocess(img_bgr)
    preds = model.predict(inp, verbose=0)
    if isinstance(preds, list):
        spoof_prob = float(preds[0][0][0])
        spoof_type_prob = preds[1][0] if len(preds) > 1 else None
    else:
        spoof_prob = float(preds[0][0])
        spoof_type_prob = None

    verdict = 'FAKE' if spoof_prob >= 0.5 else 'REAL'
    heatmap  = get_gradcam(model, inp, TARGET_LAYER)
    overlaid = overlay_heatmap(img_bgr, heatmap, alpha=0.5)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title('원본 이미지', fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(overlaid)
    color = 'red' if verdict == 'FAKE' else 'green'
    axes[1].set_title(f'Grad-CAM 오버레이\n→ {verdict} ({spoof_prob:.1%})', fontsize=12, color=color)
    axes[1].axis('off')

    heatmap_resized = cv2.resize(heatmap, (224, 224))
    axes[2].imshow(heatmap_resized, cmap='jet')
    axes[2].set_title('히트맵 (판단 집중 영역)', fontsize=12)
    axes[2].axis('off')
    plt.colorbar(axes[2].images[0], ax=axes[2], fraction=0.046)

    if spoof_type_prob is not None:
        type_names = ['Live', 'Print', 'Replay', 'Mask']
        pred_type  = type_names[spoof_type_prob.argmax()]
        plt.suptitle(
            f'{title}  |  판정: {verdict}  |  공격 유형 추정: {pred_type}',
            fontsize=13, fontweight='bold'
        )
    else:
        plt.suptitle(f'{title}  |  판정: {verdict}', fontsize=13, fontweight='bold')

    plt.tight_layout()
    safe_title = title.replace(' ', '_')
    plt.savefig(f'{REPORT_DIR}/06_gradcam_{safe_title}.png', dpi=150, bbox_inches='tight')
    plt.show()

# 각 카테고리 첫 번째 샘플로 상세 분석
for cat, info in CATEGORIES.items():
    if samples[cat]:
        analyze_single(samples[cat][0], title=info['label'])

print('상세 분석 완료')

## 7. GitHub 커밋 가이드

In [ ]:
print('생성된 파일:')
for f in Path(REPORT_DIR).glob('06_gradcam*'):
    print(f'  {f.name}')

print()
print('>>> GitHub 커밋:')
print('  git add notebooks/06_gradcam.ipynb')
print('  git add results/week13/06_gradcam_*.png')
print('  git commit -m "feat: Grad-CAM XAI visualization per attack type"')